In [ ]:
def CombinedPlot_PropertyHistogram_Perturbations(histogramsDictionary_combined, mode, zKey, time_bins,
                                                 parcelTypes=["CL", "nonCL"],
                                                 plotType="contour", normalize=True, num_levels=20,
                                                 labelsDictionary=labelsDictionaryPerturbation):
    
    mins = ModelData.time[1].item() / 1e9 / 60
    property_bins_Dictionary = GetPropertyBins_Perturbations()
    property_limits_Dictionary = GetPropertyLimits_Perturbations(property_bins_Dictionary, mode)
    varNames     = GetVarNames()
    parcelDepths = GetParcelDepths()
    colorbarLabel = "Frequency (%)" if normalize else "Count"
    cmap = plt.get_cmap("turbo").copy()
    cmap.set_under("white")

    nrows = len(parcelTypes)
    ncols = len(varNames)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.5 * ncols, 3.5 * nrows),
        constrained_layout=True
    )
    axes = np.atleast_2d(axes)

    plotObjects_by_col = {j: [] for j in range(ncols)}
    vmin_by_col = {j: np.inf  for j in range(ncols)}
    vmax_by_col = {j: -np.inf for j in range(ncols)}

    # First pass: compute global vmin/vmax per column
    for i, parcelType in enumerate(parcelTypes):
        for j, varName in enumerate(varNames):
            parcelDepth = parcelDepths[0]
            a = histogramsDictionary_combined[parcelType][parcelDepth][varName][mode][zKey]
            if normalize:
                a = NormalizeHistogram_Perturbations(a)
                a *= 100
            col_min = np.nanmin(a[:-1, :])
            col_max = np.nanmax(a[:-1, :])
            if np.isfinite(col_min):
                vmin_by_col[j] = min(vmin_by_col[j], col_min)
            if np.isfinite(col_max):
                vmax_by_col[j] = max(vmax_by_col[j], col_max)

    # Second pass: plot with shared scale
    for i, parcelType in enumerate(parcelTypes):
        for j, varName in enumerate(varNames):
            ax = axes[i, j]
            ax.set_rasterization_zorder(1) #rasterizes (removes pdf vectors) anything below zorder=1
            parcelDepth = parcelDepths[0]
            a = histogramsDictionary_combined[parcelType][parcelDepth][varName][mode][zKey]
            if normalize:
                a = NormalizeHistogram_Perturbations(a)
                a *= 100

            vmin = vmin_by_col[j]
            vmax = vmax_by_col[j]

            if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
                ax.set_visible(False)
                continue

            levels = np.linspace(vmin, vmax, num_levels)

            x = mins * time_bins
            y = property_bins_Dictionary[varName]
            x_centers = 0.5 * (x[:-1] + x[1:])
            y_centers = 0.5 * (y[:-1] + y[1:])
            X, Y = np.meshgrid(x_centers, y_centers)
            multiplier = multiplierDictionary.get(varName, 1)
            units = unitsDictionary.get(varName, "")

            if plotType == "contour":
                plotObject = ax.contourf(
                    X, multiplier * Y, a.T,
                    cmap=cmap, levels=levels,
                    vmin=vmin, vmax=vmax,extend='min'
                )
            else:
                plotObject = ax.pcolormesh(
                    x, multiplier * y, a.T,
                    cmap=cmap, shading="auto",
                    vmin=vmin, vmax=vmax
                )

            #Edit Limits
            ymin, ymax = property_limits_Dictionary[varName]
            ax.set_ylim(ymin, ymax)

            plotObjects_by_col[j].append(plotObject)

            if j == 0:
                ax.set_ylabel(f"{parcelType}\n{labelsDictionary[varName]}")
            else:
                ax.set_ylabel(fr"{labelsDictionary[varName]} ({units})")
            if i == 0:
                ax.set_title(labelsDictionary[varName])
            if i == nrows - 1:
                ax.set_xlabel("Backwards Time (mins)")

    # Add one shared colorbar per column
    for j in range(ncols):
        if not plotObjects_by_col[j]:
            continue
        plt.colorbar(
            plotObjects_by_col[j][0],
            ax=axes[:, j],
            location="right",
            shrink=0.8,
            label=colorbarLabel if j == ncols - 1 else ""
        )

    updraftType = "General" if mode == 'g' else "Cloudy"
    plt.suptitle(
        f"History of Entrained Parcels ({updraftType} Updrafts) "
        f"({zKey.replace('_', '-', 1).split('_')[0]})",
        fontsize=16
    )

    for axis in axes.flat:
        axis.axhline(0, color='black', linestyle='--', alpha=0.6, linewidth=1)
    return fig